# 04 — Entity Reconstruction, Canonical Cleaning, and LLM-Based Entity Cleaning

Pipeline stage:

- reuse existing raw NER mentions from `04A_raw_mentions`
- rebuild raw mention table from shards only if needed
- run GLiNER raw NER only when raw parquet is missing and shard coverage is incomplete, or when explicitly forced
- normalize and canonicalize entities with deterministic rules
- construct entity-level summary from canonicalized mentions
- select a large canonical candidate pool for LLM cleaning
- run resumable LLM cleaning on 10k–20k canonical entities with compact prompts
- merge LLM cleaning outputs back into the main pipeline
- rebuild final mentions, analysis mentions, summary, and context tables

Output directory

`output/04_entity_extraction/`

`04A_raw_mentions/`

- `entity_mentions_raw.parquet`
- `entity_run_config.json`
- `mention_shard_manifest.csv`
- `mention_shards/part_0000_mentions.parquet`
- `mention_shards/part_0001_mentions.parquet`
- ...

`04B_analysis_entities/`

- `entity_mentions_final.parquet`
- `entity_analysis_mentions.parquet`
- `entity_analysis_summary.parquet`
- `entity_contexts_final.parquet`
- `candidate_pool_for_llm_merge.parquet`
- `entity_llm_clean_candidates.parquet`
- `entity_llm_clean_results.jsonl`
- `entity_llm_clean_results.parquet`
- `entity_postprocess_run_config.json`

In [18]:
!pip -q install pyarrow aiohttp tqdm rapidfuzz gliner

## Environment Setup

In [2]:
import os
import re
import gc
import json
import math
import time
import asyncio
import hashlib
import warnings
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

import aiohttp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from gliner import GLiNER

from google.colab import drive

drive.mount("/content/drive")

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore", category=FutureWarning)

Mounted at /content/drive


## Paths

In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"
IN_DIR_02D = f"{BASE_DIR}/output/02D_sentence_train"
OUT_DIR_04 = f"{BASE_DIR}/output/04_entity_extraction"

RAW_DIR_04A = f"{OUT_DIR_04}/04A_raw_mentions"
ANALYSIS_DIR_04B = f"{OUT_DIR_04}/04B_analysis_entities"

CLEAN_AI_BLOCKS_PARQUET = f"{IN_DIR_02D}/clean_ai_blocks.parquet"
RAW_MENTION_SHARDS_DIR = f"{RAW_DIR_04A}/mention_shards"
RAW_MENTIONS_PARQUET = f"{RAW_DIR_04A}/entity_mentions_raw.parquet"
RAW_RUN_CONFIG_JSON = f"{RAW_DIR_04A}/entity_run_config.json"
RAW_MANIFEST_CSV = f"{RAW_DIR_04A}/mention_shard_manifest.csv"

ENTITY_MENTIONS_FINAL_PARQUET = f"{ANALYSIS_DIR_04B}/entity_mentions_final.parquet"
ENTITY_ANALYSIS_MENTIONS_PARQUET = f"{ANALYSIS_DIR_04B}/entity_analysis_mentions.parquet"
ENTITY_ANALYSIS_SUMMARY_PARQUET = f"{ANALYSIS_DIR_04B}/entity_analysis_summary.parquet"
ENTITY_CONTEXTS_FINAL_PARQUET = f"{ANALYSIS_DIR_04B}/entity_contexts_final.parquet"
CANDIDATE_POOL_FOR_LLM_PARQUET = f"{ANALYSIS_DIR_04B}/candidate_pool_for_llm_merge.parquet"

ENTITY_LLM_CANDIDATES_PARQUET = f"{ANALYSIS_DIR_04B}/entity_llm_clean_candidates.parquet"
ENTITY_LLM_RESULTS_JSONL = f"{ANALYSIS_DIR_04B}/entity_llm_clean_results.jsonl"
ENTITY_LLM_RESULTS_PARQUET = f"{ANALYSIS_DIR_04B}/entity_llm_clean_results.parquet"
ENTITY_POSTPROCESS_RUN_CONFIG_JSON = f"{ANALYSIS_DIR_04B}/entity_postprocess_run_config.json"

os.makedirs(RAW_DIR_04A, exist_ok=True)
os.makedirs(RAW_MENTION_SHARDS_DIR, exist_ok=True)
os.makedirs(ANALYSIS_DIR_04B, exist_ok=True)

assert os.path.exists(CLEAN_AI_BLOCKS_PARQUET), f"Missing clean AI blocks input: {CLEAN_AI_BLOCKS_PARQUET}"

print("CLEAN_AI_BLOCKS_PARQUET:", CLEAN_AI_BLOCKS_PARQUET)
print("RAW_MENTIONS_PARQUET:", RAW_MENTIONS_PARQUET)
print("ANALYSIS_DIR_04B:", ANALYSIS_DIR_04B)

CLEAN_AI_BLOCKS_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/02D_sentence_train/clean_ai_blocks.parquet
RAW_MENTIONS_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04A_raw_mentions/entity_mentions_raw.parquet
ANALYSIS_DIR_04B: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities


## Runtime and Configuration

In [4]:
SEED = 42
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NA"

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE)
print("gpu:", GPU_NAME)
print("torch:", torch.__version__)

FORCE_RERUN_RAW_NER = False
FORCE_REBUILD_RAW_FROM_SHARDS = False
OVERWRITE_ANALYSIS_OUTPUTS = True
REBUILD_RAW_FROM_SHARDS_IF_NEEDED = True

GLINER_MODEL_NAME = "urchade/gliner_large-v2.1"
GLINER_THRESHOLD = 0.55
NER_BATCH_SIZE = 192
NER_BLOCKS_PER_SHARD = 25000
MAX_BLOCK_CHARS_FOR_NER = 4000
NER_LABELS = [
    "company",
    "technology",
    "government institution",
    "person",
    "organization",
    "product",
    "other",
]

ANALYSIS_KEEP_TYPES = {
    "company",
    "technology",
    "government_institution",
    "person",
}

MAX_CONTEXTS_PER_ENTITY = 8
MAX_CONTEXT_CHARS = 1400
LLM_CANDIDATE_MIN_DOCS = 3
LLM_CANDIDATE_MIN_ROWS = 5
LLM_CANDIDATE_TARGET_N = 15000
LLM_CANDIDATE_HARD_CAP = 18000

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_ENDPOINT = f"{DEEPSEEK_BASE_URL}/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"

WORKERS = 96
QUEUE_SIZE = 512
HTTP_LIMIT = 192
HTTP_PER_HOST = 96
HTTP_TIMEOUT = 90
MAX_RETRIES = 6
JSONL_FLUSH_EVERY = 100

LLM_MAX_CONTEXT_EXAMPLES = 3
LLM_CONTEXT_SNIPPET_CHARS = 320

MIN_CONFIDENCE_KEEP = 0.45
MIN_CANONICAL_LEN = 2
MAX_CANONICAL_LEN = 120

NEWS_SOURCE_TERMS = {
    "abc news", "afp", "agence france-presse", "al jazeera", "ap", "ars technica",
    "associated press", "axios", "bbc", "bbc news", "bloomberg", "boston globe",
    "business insider", "business wire", "buzzfeed", "cbs news", "chicago tribune",
    "cnbc", "cnet", "cnn", "daily mail", "deutsche welle", "dw", "economist",
    "financial times", "forbes", "fox news", "fortune", "guardian", "huffpost",
    "independent", "insider", "marketwatch", "msnbc", "nbc news", "new york post",
    "new york times", "newsweek", "npr", "politico", "pr newswire", "prnewswire",
    "reuters", "sky news", "techcrunch", "the associated press", "the atlantic",
    "the economist", "the guardian", "the information", "the intercept",
    "the new york times", "the wall street journal", "the washington post",
    "time", "usa today", "vice", "vox", "wall street journal", "washington post",
    "wired", "wsj", "xinhua", "yahoo finance", "yahoo news", "zdnet"
}

GENERIC_PERSON_TERMS = {
    "he", "she", "they", "we", "i", "you", "users", "researchers", "scientists",
    "developers", "engineers", "students", "workers", "people", "consumers",
    "customers", "patients", "teachers", "parents", "children", "officials",
    "executives", "lawmakers", "analysts", "experts", "critics", "authors",
    "journalists", "publishers", "creators", "employees", "employers",
    "founders", "investors"
}

GENERIC_GEO_TERMS = {
    "world", "europe", "asia", "africa", "america", "americas", "global",
    "international"
}

GENERIC_TECH_TERMS = {
    "ai", "artificial intelligence", "machine learning", "deep learning",
    "generative ai", "large language model", "large language models",
    "llm", "llms", "chatbot", "chatbots", "ai model", "ai models",
    "ai tool", "ai tools", "ai system", "ai systems", "ai technology",
    "ai technologies"
}

TYPE_PRIORITY = {
    "company": 6,
    "technology": 5,
    "government_institution": 4,
    "person": 3,
    "product": 2,
    "organization": 1,
    "other": 0,
}

TYPE_NORMALIZATION_MAP = {
    "government institution": "government_institution",
    "government_institution": "government_institution",
    "location": "government_institution",
    "person": "person",
    "company": "company",
    "technology": "technology",
    "organization": "organization",
    "product": "product",
    "other": "other",
}

SAFE_CANONICAL_MAP = {
    "open ai": "OpenAI",
    "chat gpt": "ChatGPT",
    "gpt 4": "GPT-4",
    "gpt4": "GPT-4",
    "gpt 4o": "GPT-4o",
    "gpt4o": "GPT-4o",
    "nvidia": "Nvidia",
    "u s": "United States",
    "u s a": "United States",
    "usa": "United States",
    "u k": "United Kingdom",
    "uk": "United Kingdom",
    "eu": "European Union",
    "a i": "AI",
    "ai": "AI",
    "a i models": "AI models",
    "a i systems": "AI systems",
    "a i tools": "AI tools",
    "a i technology": "AI technology",
    "a i technologies": "AI technologies",
}

TYPE_HINTS = {
    "openai": "company",
    "google": "company",
    "microsoft": "company",
    "amazon": "company",
    "apple": "company",
    "nvidia": "company",
    "meta": "company",
    "anthropic": "company",
    "alphabet": "company",
    "tesla": "company",
    "chatgpt": "technology",
    "gpt-4": "technology",
    "gpt-4o": "technology",
    "claude": "technology",
    "gemini": "technology",
    "copilot": "technology",
    "ai": "technology",
    "artificial intelligence": "technology",
    "generative ai": "technology",
    "machine learning": "technology",
    "large language model": "technology",
    "large language models": "technology",
    "united states": "government_institution",
    "china": "government_institution",
    "india": "government_institution",
    "united kingdom": "government_institution",
    "european union": "government_institution",
    "california": "government_institution",
    "new york": "government_institution",
    "san francisco": "government_institution",
}

run_config = {
    "seed": SEED,
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "force_rerun_raw_ner": FORCE_RERUN_RAW_NER,
    "force_rebuild_raw_from_shards": FORCE_REBUILD_RAW_FROM_SHARDS,
    "overwrite_analysis_outputs": OVERWRITE_ANALYSIS_OUTPUTS,
    "rebuild_raw_from_shards_if_needed": REBUILD_RAW_FROM_SHARDS_IF_NEEDED,
    "gliner_model_name": GLINER_MODEL_NAME,
    "gliner_threshold": GLINER_THRESHOLD,
    "ner_labels": NER_LABELS,
    "ner_batch_size": NER_BATCH_SIZE,
    "ner_blocks_per_shard": NER_BLOCKS_PER_SHARD,
    "max_block_chars_for_ner": MAX_BLOCK_CHARS_FOR_NER,
    "analysis_keep_types": sorted(ANALYSIS_KEEP_TYPES),
    "max_contexts_per_entity": MAX_CONTEXTS_PER_ENTITY,
    "max_context_chars": MAX_CONTEXT_CHARS,
    "llm_candidate_min_docs": LLM_CANDIDATE_MIN_DOCS,
    "llm_candidate_min_rows": LLM_CANDIDATE_MIN_ROWS,
    "llm_candidate_target_n": LLM_CANDIDATE_TARGET_N,
    "llm_candidate_hard_cap": LLM_CANDIDATE_HARD_CAP,
    "deepseek_model": DEEPSEEK_MODEL,
    "workers": WORKERS,
    "queue_size": QUEUE_SIZE,
    "http_limit": HTTP_LIMIT,
    "http_per_host": HTTP_PER_HOST,
    "http_timeout": HTTP_TIMEOUT,
    "max_retries": MAX_RETRIES,
    "jsonl_flush_every": JSONL_FLUSH_EVERY,
    "llm_max_context_examples": LLM_MAX_CONTEXT_EXAMPLES,
    "llm_context_snippet_chars": LLM_CONTEXT_SNIPPET_CHARS,
    "min_confidence_keep": MIN_CONFIDENCE_KEEP,
    "min_canonical_len": MIN_CANONICAL_LEN,
    "max_canonical_len": MAX_CANONICAL_LEN,
}

with open(RAW_RUN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)

with open(ENTITY_POSTPROCESS_RUN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)

print("wrote:", RAW_RUN_CONFIG_JSON)
print("wrote:", ENTITY_POSTPROCESS_RUN_CONFIG_JSON)

device: cuda
gpu: NVIDIA A100-SXM4-80GB
torch: 2.10.0+cu128
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04A_raw_mentions/entity_run_config.json
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_postprocess_run_config.json


## Utility Functions

In [5]:
def normalize_spaces(x: str) -> str:
    return re.sub(r"\s+", " ", str(x or "")).strip()


def strip_outer_punct(x: str) -> str:
    x = str(x or "").strip()
    x = x.strip(" \t\r\n-–—:;,.\'\"“”‘’()[]{}<>|/")
    return x


def split_camel(x: str) -> str:
    return normalize_spaces(re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(x or "")))


def pre_ner_text(x: str) -> str:
    return normalize_spaces(str(x or "")[:MAX_BLOCK_CHARS_FOR_NER])


def simple_title_case(x: str) -> str:
    words = []
    for w in str(x or "").split():
        if w.isupper() and len(w) <= 4:
            words.append(w)
        elif re.fullmatch(r"[A-Za-z]+", w):
            words.append(w.capitalize())
        else:
            words.append(w)
    return " ".join(words)


def sha1_text(text: str) -> str:
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()


def normalize_type(x: str) -> str:
    k = normalize_spaces(str(x or "")).lower().replace("-", " ").replace("/", " ")
    k = re.sub(r"\s+", " ", k)
    return TYPE_NORMALIZATION_MAP.get(k, "other")


def normalize_entity_key(x: str) -> str:
    s = normalize_spaces(split_camel(str(x or "")).lower())
    s = re.sub(r"['’]", "", s)
    s = re.sub(r"[\(\)\[\]\{\}\.,;:!?\"“”‘’/\\|]+", " ", s)
    s = re.sub(r"[-_]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def canonicalize_surface(surface: str) -> str:
    s = normalize_spaces(split_camel(surface))
    s = strip_outer_punct(s)
    s = re.sub(r"^(the|a|an)\s+", "", s, flags=re.IGNORECASE)
    s = re.sub(r"(?:'s|’s)$", "", s, flags=re.IGNORECASE)
    s = normalize_spaces(s)
    key = normalize_entity_key(s)
    if key in SAFE_CANONICAL_MAP:
        return SAFE_CANONICAL_MAP[key]
    if re.fullmatch(r"[A-Z]{2,6}", s):
        return s
    if key in GENERIC_TECH_TERMS:
        if key == "large language models":
            return "Large Language Models"
        return simple_title_case(key)
    return simple_title_case(s)


def repair_type(canonical_entity: str, final_type_raw: str) -> str:
    key = normalize_entity_key(canonical_entity)
    if key in TYPE_HINTS:
        return TYPE_HINTS[key]
    t = normalize_type(final_type_raw)
    if key in GENERIC_PERSON_TERMS:
        return "person"
    if key in GENERIC_GEO_TERMS:
        return "government_institution"
    return t


def is_numeric_junk_entity(x: str) -> bool:
    s = normalize_spaces(str(x or ""))
    if s == "":
        return True
    if len(s) <= 1:
        return True
    if re.fullmatch(r"[#\$€£¥]?\d[\d,.\-+%/ ]*", s):
        return True
    if re.fullmatch(r"[#\$€£¥]?[A-Z]*\d+[A-Z0-9\-+./% ]*", s):
        return True
    if re.fullmatch(r"(#|No\.?|Booth|Stand)\s*[A-Z0-9.\-]+", s, flags=re.IGNORECASE):
        return True
    if re.fullmatch(r"[$€£¥]\s*\d[\d,.\-+% ]*", s):
        return True
    return False


def is_news_source_entity(canonical_entity: str) -> bool:
    return normalize_entity_key(canonical_entity) in NEWS_SOURCE_TERMS


def is_generic_pronoun_or_actor(canonical_entity: str, final_type: str) -> bool:
    key = normalize_entity_key(canonical_entity)
    if key in GENERIC_PERSON_TERMS:
        return True
    if final_type == "person" and key in GENERIC_PERSON_TERMS:
        return True
    return False


def is_low_value_government_entity(canonical_entity: str) -> bool:
    key = normalize_entity_key(canonical_entity)
    if key in GENERIC_GEO_TERMS:
        return True
    if re.fullmatch(r"#?\d[\w.\-+]*", key):
        return True
    return False


def compact_text_snippet(x: str, max_chars: int) -> str:
    s = normalize_spaces(x)
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + " ..."


def choose_group_type(types: List[str]) -> str:
    cnt = Counter(types)
    ranked = sorted(cnt.items(), key=lambda kv: (kv[1], TYPE_PRIORITY.get(kv[0], -1)), reverse=True)
    return ranked[0][0] if ranked else "other"


def jsonl_read_existing(path: str) -> Dict[str, Dict[str, Any]]:
    done = {}
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                ent = obj.get("canonical_entity")
                typ = obj.get("final_type")
                if ent is None or typ is None:
                    continue
                done[f"{ent}|||{typ}"] = obj
            except Exception:
                continue
    return done


def parquet_exists_and_nonempty(path: str) -> bool:
    return os.path.exists(path) and os.path.getsize(path) > 0


def validate_raw_schema(df: pd.DataFrame) -> None:
    required = [
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len", "entity_surface",
        "entity_surface_norm", "entity_key", "raw_label", "raw_score", "char_start", "char_end"
    ]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing required raw mention columns: {missing}"


def empty_raw_mentions_frame() -> pd.DataFrame:
    cols = [
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len", "entity_surface",
        "entity_surface_norm", "entity_key", "raw_label", "raw_score", "char_start", "char_end"
    ]
    return pd.DataFrame(columns=cols)


def get_shard_path(shard_id: int) -> str:
    return os.path.join(RAW_MENTION_SHARDS_DIR, f"part_{shard_id:04d}_mentions.parquet")


def load_clean_ai_blocks() -> pd.DataFrame:
    blocks = pd.read_parquet(CLEAN_AI_BLOCKS_PARQUET)
    required_cols = {"doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain", "clean_block_text", "clean_block_len"}
    missing = required_cols - set(blocks.columns)
    assert not missing, f"Missing required block columns in clean_ai_blocks.parquet: {sorted(missing)}"
    if "p_ai_block" not in blocks.columns:
        blocks["p_ai_block"] = np.nan
    keep_cols = [
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len"
    ]
    blocks = blocks[keep_cols].copy()
    blocks["clean_block_text"] = blocks["clean_block_text"].fillna("").astype(str)
    blocks["block_uid"] = blocks["block_uid"].astype(str)
    blocks = blocks[blocks["clean_block_text"].str.len() > 0].drop_duplicates(["block_uid"]).reset_index(drop=True)
    return blocks


def load_or_init_manifest() -> pd.DataFrame:
    if os.path.exists(RAW_MANIFEST_CSV):
        manifest = pd.read_csv(RAW_MANIFEST_CSV)
    else:
        manifest = pd.DataFrame(columns=["shard_id", "out_path", "n_blocks", "n_mentions", "status"])
    expected_cols = ["shard_id", "out_path", "n_blocks", "n_mentions", "status"]
    for col in expected_cols:
        if col not in manifest.columns:
            manifest[col] = pd.Series(dtype="object")
    return manifest[expected_cols].copy()


def save_manifest(manifest: pd.DataFrame) -> None:
    manifest = manifest.sort_values("shard_id").reset_index(drop=True)
    manifest.to_csv(RAW_MANIFEST_CSV, index=False)


def extract_entities_from_frame(model: GLiNER, pdf: pd.DataFrame) -> pd.DataFrame:
    texts = [pre_ner_text(x) for x in pdf["clean_block_text"].tolist()]
    preds = model.batch_predict_entities(texts, NER_LABELS, threshold=GLINER_THRESHOLD)
    records = []
    for row, ents in zip(pdf.itertuples(index=False), preds):
        for ent in ents:
            surface = str(ent.get("text", "")).strip()
            if not surface:
                continue
            raw_label = str(ent.get("label", "")).strip().lower()
            score = float(ent.get("score", 0.0))
            start = ent.get("start", None)
            end = ent.get("end", None)
            records.append({
                "doc_id": int(row.doc_id),
                "block_id": int(row.block_id),
                "block_uid": str(row.block_uid),
                "url": row.url,
                "date": row.date,
                "language": row.language,
                "title": row.title,
                "domain": row.domain,
                "clean_block_text": row.clean_block_text,
                "p_ai_block": row.p_ai_block,
                "clean_block_len": int(row.clean_block_len),
                "entity_surface": surface,
                "entity_surface_norm": canonicalize_surface(surface),
                "entity_key": normalize_entity_key(surface),
                "raw_label": raw_label,
                "raw_score": score,
                "char_start": int(start) if start is not None else -1,
                "char_end": int(end) if end is not None else -1,
            })
    if not records:
        return empty_raw_mentions_frame()
    out = pd.DataFrame(records)
    return out[[
        "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
        "clean_block_text", "p_ai_block", "clean_block_len", "entity_surface",
        "entity_surface_norm", "entity_key", "raw_label", "raw_score", "char_start", "char_end"
    ]]

## Raw NER Reuse / Merge / GLiNER Extraction

In [6]:
blocks = load_clean_ai_blocks()
print("clean_ai_blocks shape:", blocks.shape)
display(blocks.head(3))

planned_shards = []
for shard_id, start_row in enumerate(range(0, len(blocks), NER_BLOCKS_PER_SHARD)):
    end_row = min(start_row + NER_BLOCKS_PER_SHARD, len(blocks))
    planned_shards.append((shard_id, start_row, end_row))

manifest = load_or_init_manifest()
existing_shard_files = sorted([f for f in os.listdir(RAW_MENTION_SHARDS_DIR) if f.endswith(".parquet")])
existing_shard_ids = {int(re.search(r"part_(\d+)_mentions\.parquet$", x).group(1)) for x in existing_shard_files if re.search(r"part_(\d+)_mentions\.parquet$", x)}
planned_shard_ids = {x[0] for x in planned_shards}
missing_shard_ids = sorted(planned_shard_ids - existing_shard_ids)

need_raw_ner = FORCE_RERUN_RAW_NER or ((not parquet_exists_and_nonempty(RAW_MENTIONS_PARQUET)) and len(missing_shard_ids) > 0)
need_merge_only = FORCE_REBUILD_RAW_FROM_SHARDS or ((not parquet_exists_and_nonempty(RAW_MENTIONS_PARQUET)) and len(missing_shard_ids) == 0)

print("planned shards:", len(planned_shards))
print("existing shard files:", len(existing_shard_ids))
print("missing shard ids:", len(missing_shard_ids))
print("need_raw_ner:", need_raw_ner)
print("need_merge_only:", need_merge_only)

if need_raw_ner:
    assert DEVICE == "cuda", "GPU is required for GLiNER raw extraction."
    ner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME).to(DEVICE)

    for shard_id, start_row, end_row in tqdm(planned_shards, desc="GLiNER shard extraction"):
        out_path = get_shard_path(shard_id)
        if (not FORCE_RERUN_RAW_NER) and os.path.exists(out_path):
            continue

        shard_df = blocks.iloc[start_row:end_row].copy().reset_index(drop=True)
        all_out = []
        n_batches = math.ceil(len(shard_df) / NER_BATCH_SIZE)

        for batch_idx in tqdm(range(n_batches), total=n_batches, desc=f"part_{shard_id:04d}", leave=False):
            sub = shard_df.iloc[batch_idx * NER_BATCH_SIZE:(batch_idx + 1) * NER_BATCH_SIZE].copy()
            part_out = extract_entities_from_frame(ner_model, sub)
            if len(part_out) > 0:
                all_out.append(part_out)

        shard_mentions = pd.concat(all_out, ignore_index=True) if all_out else empty_raw_mentions_frame()
        shard_mentions.to_parquet(out_path, index=False)

        manifest = manifest.loc[manifest["shard_id"] != shard_id].copy()
        manifest = pd.concat([
            manifest,
            pd.DataFrame([{
                "shard_id": shard_id,
                "out_path": out_path,
                "n_blocks": len(shard_df),
                "n_mentions": len(shard_mentions),
                "status": "ok",
            }])
        ], ignore_index=True)
        save_manifest(manifest)

        del shard_df, all_out, shard_mentions
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del ner_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if need_raw_ner or need_merge_only:
    shard_files = sorted([
        os.path.join(RAW_MENTION_SHARDS_DIR, f)
        for f in os.listdir(RAW_MENTION_SHARDS_DIR)
        if f.endswith(".parquet")
    ])
    assert shard_files, f"No raw mention shards found in: {RAW_MENTION_SHARDS_DIR}"

    frames = []
    for fpath in tqdm(shard_files, desc="Loading raw mention shards"):
        frames.append(pd.read_parquet(fpath))
    raw_mentions = pd.concat(frames, ignore_index=True) if frames else empty_raw_mentions_frame()
    del frames
    gc.collect()

    validate_raw_schema(raw_mentions)
    raw_mentions.to_parquet(RAW_MENTIONS_PARQUET, index=False)
    print("rebuilt raw mentions to:", RAW_MENTIONS_PARQUET)
else:
    assert parquet_exists_and_nonempty(RAW_MENTIONS_PARQUET), (
        "Missing raw mention parquet and shard rebuild path is unavailable."
    )
    raw_mentions = pd.read_parquet(RAW_MENTIONS_PARQUET)
    print("loaded raw mentions from:", RAW_MENTIONS_PARQUET)

validate_raw_schema(raw_mentions)
manifest = load_or_init_manifest()
print("raw_mentions shape:", raw_mentions.shape)
print("manifest shape:", manifest.shape)
display(raw_mentions.head(5))
display(manifest.head(10))

clean_ai_blocks shape: (731989, 11)


,doc_id,block_id,block_uid,url,date,language,title,domain,clean_block_text,p_ai_block,clean_block_len
0,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,1977
1,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542
2,1,8,1:8,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,Imagine showing these to a tribe that's never ...,0.998448,557


planned shards: 30
existing shard files: 30
missing shard ids: 0
need_raw_ner: False
need_merge_only: False
loaded raw mentions from: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04A_raw_mentions/entity_mentions_raw.parquet
raw_mentions shape: (5113052, 18)
manifest shape: (30, 5)


,doc_id,block_id,block_uid,url,date,language,title,domain,clean_block_text,p_ai_block,clean_block_len,entity_surface,entity_surface_norm,entity_key,raw_label,raw_score,char_start,char_end
0,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,1977,Byron Gilliam,Byron Gilliam,byron gilliam,person,0.903304,1035,1048
1,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,1977,Carlos Gonzalez Campo,Carlos Gonzalez Campo,carlos gonzalez campo,person,0.646446,1447,1468
2,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542,AI,AI,ai,technology,0.811913,70,72
3,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542,AI,AI,ai,technology,0.597019,101,103
4,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,542,AI,AI,ai,technology,0.868772,175,177


,shard_id,out_path,n_blocks,n_mentions,status
0,0,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,174687,ok
1,1,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,174681,ok
2,2,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,171504,ok
3,3,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,175129,ok
4,4,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,173717,ok
5,5,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,172515,ok
6,6,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,174042,ok
7,7,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,175265,ok
8,8,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,175312,ok
9,9,/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_C...,NaN,172452,ok


## Deterministic Canonical Cleaning

In [7]:
mentions = raw_mentions.copy()

mentions["final_type_raw"] = mentions["raw_label"].map(normalize_type)
mentions["confidence"] = pd.to_numeric(mentions["raw_score"], errors="coerce").astype(float)

mentions["entity_surface"] = mentions["entity_surface"].fillna("").astype(str)
mentions["entity_surface_norm"] = mentions["entity_surface_norm"].fillna("").astype(str)
mentions["entity_key"] = mentions["entity_key"].fillna("").astype(str)
mentions["block_text_hash"] = mentions["clean_block_text"].fillna("").astype(str).map(sha1_text)

mentions["canonical_entity"] = mentions["entity_surface_norm"].where(
    mentions["entity_surface_norm"].str.strip() != "",
    mentions["entity_surface"]
)
mentions["canonical_entity"] = mentions["canonical_entity"].map(canonicalize_surface)
mentions["canonical_key"] = mentions["canonical_entity"].map(normalize_entity_key)
mentions["final_type"] = [repair_type(ent, t) for ent, t in zip(mentions["canonical_entity"], mentions["final_type_raw"])]

mentions["is_news_source"] = mentions["canonical_entity"].map(is_news_source_entity)
mentions["is_numeric_junk"] = mentions["canonical_entity"].map(is_numeric_junk_entity)
mentions["is_generic_actor"] = [is_generic_pronoun_or_actor(ent, t) for ent, t in zip(mentions["canonical_entity"], mentions["final_type"])]
mentions["is_low_value_gov"] = [is_low_value_government_entity(ent) if t == "government_institution" else False for ent, t in zip(mentions["canonical_entity"], mentions["final_type"])]

mentions = mentions[
    (mentions["confidence"] >= MIN_CONFIDENCE_KEEP)
    & (~mentions["is_news_source"])
    & (~mentions["is_numeric_junk"])
].copy()

mentions = mentions[mentions["canonical_entity"].str.len().between(MIN_CANONICAL_LEN, MAX_CANONICAL_LEN)].copy()
mentions = mentions[mentions["canonical_key"].str.len().between(MIN_CANONICAL_LEN, MAX_CANONICAL_LEN)].copy()
mentions = mentions.drop_duplicates(["block_uid", "canonical_key", "final_type", "char_start", "char_end"]).reset_index(drop=True)

print("mentions after deterministic cleaning:", mentions.shape)
display(mentions[[
    "entity_surface", "canonical_entity", "final_type_raw", "final_type",
    "confidence", "is_news_source", "is_numeric_junk"
]].head(20))

mentions after deterministic cleaning: (5057060, 28)


,entity_surface,canonical_entity,final_type_raw,final_type,confidence,is_news_source,is_numeric_junk
0,Byron Gilliam,Byron Gilliam,person,person,0.903304,False,False
1,Carlos Gonzalez Campo,Carlos Gonzalez Campo,person,person,0.646446,False,False
2,AI,AI,technology,technology,0.811913,False,False
3,AI,AI,technology,technology,0.597019,False,False
4,AI,AI,technology,technology,0.868772,False,False
5,Werners AI Art,Werners AI Art,organization,organization,0.856225,False,False
6,Ai,AI,technology,technology,0.900137,False,False
7,AI,AI,technology,technology,0.963050,False,False
8,AI,AI,technology,technology,0.940707,False,False
9,AI,AI,technology,technology,0.995971,False,False


## Initial Canonical Summary Before LLM Cleaning

In [8]:
summary_pre = (
    mentions.groupby(["canonical_entity", "final_type", "canonical_key"], dropna=False)
    .agg(
        n_docs=("doc_id", "nunique"),
        n_rows=("canonical_entity", "size"),
        n_domains=("domain", "nunique"),
        confidence_mean=("confidence", "mean"),
    )
    .reset_index()
    .sort_values(["n_docs", "n_rows", "n_domains", "confidence_mean"], ascending=[False, False, False, False])
    .reset_index(drop=True)
)
summary_pre["keep_for_analysis"] = summary_pre["final_type"].isin(ANALYSIS_KEEP_TYPES)

print("summary_pre shape:", summary_pre.shape)
display(summary_pre.head(30))

summary_pre shape: (458681, 8)


,canonical_entity,final_type,canonical_key,n_docs,n_rows,n_domains,confidence_mean,keep_for_analysis
0,AI,technology,ai,112374,730428,4673,0.880893,True
1,Artificial Intelligence,technology,artificial intelligence,52820,99056,3803,0.821245,True
2,OpenAI,company,open ai,28616,95960,2316,0.806287,True
3,Google,company,google,26404,95115,2435,0.831170,True
4,Microsoft,company,microsoft,22494,74173,2130,0.834231,True
5,ChatGPT,product,chat gpt,21637,59950,2359,0.795421,False
6,United States,government_institution,united states,16964,31739,2054,0.764726,True
7,ChatGPT,technology,chat gpt,15114,44034,2363,0.759445,True
8,OpenAI,organization,open ai,14881,44597,1894,0.762220,False
9,Generative Ai,technology,generative ai,12127,35582,1609,0.830927,True


## Build Compact Candidate Pool for LLM Cleaning

In [9]:
candidate_pool = summary_pre[
    (summary_pre["n_docs"] >= LLM_CANDIDATE_MIN_DOCS)
    | (summary_pre["n_rows"] >= LLM_CANDIDATE_MIN_ROWS)
].copy()

DROP_PATTERNS = [r"^#", r"^\$", r"\d{3,}", r"\bbooth\b"]

def cheap_filter(name: str) -> bool:
    name = str(name).lower()
    for p in DROP_PATTERNS:
        if re.search(p, name):
            return False
    return True

before = len(candidate_pool)
candidate_pool = candidate_pool[candidate_pool["canonical_entity"].apply(cheap_filter)].copy()
after = len(candidate_pool)
print(f"cheap filter removed {before - after} entities")

candidate_pool["generic_penalty"] = candidate_pool["canonical_key"].isin(GENERIC_TECH_TERMS).astype(int)
candidate_pool["actor_penalty"] = candidate_pool["canonical_key"].isin(GENERIC_PERSON_TERMS).astype(int)
candidate_pool["priority_score"] = (
    candidate_pool["n_docs"] * 5
    + candidate_pool["n_domains"] * 2
    + np.log1p(candidate_pool["n_rows"]) * 10
    - candidate_pool["generic_penalty"] * 5
    - candidate_pool["actor_penalty"] * 5
)

candidate_pool = candidate_pool.sort_values(
    ["priority_score", "n_docs", "n_domains", "n_rows", "confidence_mean"],
    ascending=[False, False, False, False, False]
).reset_index(drop=True)

if len(candidate_pool) > LLM_CANDIDATE_HARD_CAP:
    candidate_pool = candidate_pool.head(LLM_CANDIDATE_HARD_CAP).copy()
if len(candidate_pool) > LLM_CANDIDATE_TARGET_N:
    candidate_pool = candidate_pool.head(LLM_CANDIDATE_TARGET_N).copy()

candidate_pool["llm_record_id"] = np.arange(len(candidate_pool)).astype(int)

candidate_pool.to_parquet(ENTITY_LLM_CANDIDATES_PARQUET, index=False)
candidate_pool[[
    "canonical_entity", "final_type", "n_docs", "n_rows",
    "n_domains", "confidence_mean", "keep_for_analysis", "canonical_key"
]].to_parquet(CANDIDATE_POOL_FOR_LLM_PARQUET, index=False)

print("candidate_pool shape:", candidate_pool.shape)
print("wrote:", ENTITY_LLM_CANDIDATES_PARQUET)
print("wrote:", CANDIDATE_POOL_FOR_LLM_PARQUET)
display(candidate_pool.head(30))

cheap filter removed 710 entities
candidate_pool shape: (15000, 12)
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_llm_clean_candidates.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/candidate_pool_for_llm_merge.parquet


,canonical_entity,final_type,canonical_key,n_docs,n_rows,n_domains,confidence_mean,keep_for_analysis,generic_penalty,actor_penalty,priority_score,llm_record_id
0,AI,technology,ai,112374,730428,4673,0.880893,True,1,0,571346.013873,0
1,Artificial Intelligence,technology,artificial intelligence,52820,99056,3803,0.821245,True,1,0,271816.034507,1
2,OpenAI,company,open ai,28616,95960,2316,0.806287,True,0,0,147826.716971,2
3,Google,company,google,26404,95115,2435,0.831170,True,0,0,137004.628525,3
4,Microsoft,company,microsoft,22494,74173,2130,0.834231,True,0,0,116842.141690,4
5,ChatGPT,product,chat gpt,21637,59950,2359,0.795421,False,0,0,113013.012828,5
6,United States,government_institution,united states,16964,31739,2054,0.764726,True,0,0,89031.653330,6
7,ChatGPT,technology,chat gpt,15114,44034,2363,0.759445,True,0,0,80402.927401,7
8,OpenAI,organization,open ai,14881,44597,1894,0.762220,False,0,0,78300.054443,8
9,Generative Ai,technology,generative ai,12127,35582,1609,0.830927,True,1,0,63952.796233,9


## Build Compact Context Examples for LLM Cleaning

In [10]:
candidate_keys = set(candidate_pool["canonical_key"].tolist())
candidate_types = set(candidate_pool["final_type"].tolist())

candidate_mentions = mentions[
    mentions["canonical_key"].isin(candidate_keys)
    & mentions["final_type"].isin(candidate_types)
].copy()

context_rows = []
for (canonical_entity, final_type, canonical_key), sub in tqdm(
    candidate_mentions.groupby(["canonical_entity", "final_type", "canonical_key"], dropna=False),
    desc="Building compact candidate contexts"
):
    sub = sub.sort_values(["confidence", "clean_block_len", "doc_id", "block_id"], ascending=[False, False, True, True])
    sub = sub.drop_duplicates(["doc_id", "block_id", "block_text_hash"])
    topk = sub.head(LLM_MAX_CONTEXT_EXAMPLES)

    contexts = []
    for row in topk.itertuples(index=False):
        contexts.append({
            "domain": row.domain,
            "title": compact_text_snippet(row.title, 120),
            "text": compact_text_snippet(row.clean_block_text, LLM_CONTEXT_SNIPPET_CHARS),
            "confidence": round(float(row.confidence), 4),
        })
    context_rows.append({
        "canonical_entity": canonical_entity,
        "final_type": final_type,
        "canonical_key": canonical_key,
        "contexts_compact": contexts,
    })

candidate_contexts = pd.DataFrame(context_rows)
candidate_pool = candidate_pool.merge(candidate_contexts, on=["canonical_entity", "final_type", "canonical_key"], how="left")
candidate_pool["contexts_compact"] = candidate_pool["contexts_compact"].apply(lambda x: x if isinstance(x, list) else [])
candidate_pool.to_parquet(ENTITY_LLM_CANDIDATES_PARQUET, index=False)

print("updated candidate pool with contexts:", ENTITY_LLM_CANDIDATES_PARQUET)
display(candidate_pool[["canonical_entity", "final_type", "n_docs", "contexts_compact"]].head(5))

Building compact candidate contexts:   0%|          | 0/25358 [00:00<?, ?it/s]

updated candidate pool with contexts: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_llm_clean_candidates.parquet


,canonical_entity,final_type,n_docs,contexts_compact
0,AI,technology,112374,"[{'domain': 'forbes.com', 'title': '17 Ways To..."
1,Artificial Intelligence,technology,52820,"[{'domain': 'techrepublic.com', 'title': 'Top ..."
2,OpenAI,company,28616,"[{'domain': 'globalnews.ca', 'title': 'News pu..."
3,Google,company,26404,"[{'domain': 'trend.az', 'title': 'Georgia part..."
4,Microsoft,company,22494,"[{'domain': 'kesq.com', 'title': 'The maker of..."


## Resumable LLM Cleaning on Canonical Entities

In [11]:
api_key = None
if userdata is not None:
    try:
        api_key = userdata.get("DEEPSEEK_API_KEY")
    except Exception:
        api_key = None

assert api_key, "Missing DEEPSEEK_API_KEY in Colab secrets."

existing_llm = jsonl_read_existing(ENTITY_LLM_RESULTS_JSONL)
print("existing llm-clean results:", len(existing_llm))

llm_jobs = []
for row in candidate_pool.itertuples(index=False):
    job_key = f"{row.canonical_entity}|||{row.final_type}"
    if job_key in existing_llm:
        continue
    llm_jobs.append({
        "canonical_entity": row.canonical_entity,
        "canonical_key": row.canonical_key,
        "final_type": row.final_type,
        "n_docs": int(row.n_docs),
        "n_rows": int(row.n_rows),
        "n_domains": int(row.n_domains),
        "confidence_mean": float(row.confidence_mean),
        "contexts_compact": row.contexts_compact if isinstance(row.contexts_compact, list) else [],
    })

print("pending llm-clean jobs:", len(llm_jobs))

SYSTEM_PROMPT = """
Return exactly one JSON object and nothing else.

Task:
Decide whether a canonical entity should be kept for downstream analysis in an AI-industry news project.

Primary objective:
Keep only HIGH-VALUE entities that are specific, analyzable, and useful for downstream entity-level analysis.
When uncertain, prefer DROP over KEEP.

High-value entities to KEEP:
1. Companies and business entities directly relevant to AI, semiconductors, cloud, platforms, media-tech, robotics, cybersecurity, enterprise software, healthcare AI, finance AI, or adjacent AI industries.
2. Technologies, models, platforms, products, and technical concepts that are specific and analyzable.
3. Government or regulatory bodies, countries, unions, agencies, legislatures, and courts relevant to AI policy or geopolitics.
4. Specific persons who are decision-makers, founders, CEOs, researchers, politicians, or public figures materially tied to AI topics.

Always DROP:
1. News publishers, media outlets, wires, magazines, newspapers, TV channels, podcasts, newsletters.
2. Generic actors or vague human groups.
3. Generic non-specific terms that are too broad for entity analysis.
4. Low-value locations unless they are functioning as major geopolitical/regulatory entities.
5. Booth numbers, venue markers, floor labels, addresses, room numbers, stock tickers without clear entity identity, hashtags, promo phrases, event slogans, malformed strings, numeric junk.
6. Purely promotional or catalog-like strings.
7. Duplicate spelling variants that should normalize to an existing canonical form.

Type policy:
- company
- technology
- government_institution
- person
- organization
- product
- other
- drop

Output fields:
- canonical_entity: string
- final_type_input: string
- keep_for_analysis_llm: boolean
- cleaned_type_llm: one of [company, technology, government_institution, person, organization, product, other, drop]
- cleaned_name_llm: string
- drop_reason_llm: one of [keep, news_source, numeric_junk, pronoun_generic_actor, generic_non_specific, low_value_location, malformed, duplicate_variant, unclear]
- confidence_flag_llm: one of [high, medium, low]
- notes_llm: short string <= 12 words

Important:
- If cleaned_type_llm="drop", then keep_for_analysis_llm must be false.
- If entity is generic, noisy, malformed, or borderline, drop it.
- Return JSON only.
""".strip()


def build_user_prompt(job: Dict[str, Any]) -> str:
    payload = {
        "canonical_entity": job["canonical_entity"],
        "canonical_key": job["canonical_key"],
        "final_type_input": job["final_type"],
        "n_docs": job["n_docs"],
        "n_rows": job["n_rows"],
        "n_domains": job["n_domains"],
        "confidence_mean": round(float(job["confidence_mean"]), 4),
        "shape_signals": {
            "has_hash_prefix": str(job["canonical_entity"]).strip().startswith("#"),
            "has_dollar_prefix": str(job["canonical_entity"]).strip().startswith("$"),
            "contains_digit": any(ch.isdigit() for ch in str(job["canonical_entity"])),
            "token_count": len(str(job["canonical_entity"]).split()),
            "is_all_caps": str(job["canonical_entity"]).isupper(),
        },
        "contexts": job["contexts_compact"][:4],
    }
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


def parse_json_object(text: str) -> Dict[str, Any]:
    text = str(text or "").strip()
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found.")
    return json.loads(m.group(0))


async def call_deepseek(session: aiohttp.ClientSession, job: Dict[str, Any]) -> Dict[str, Any]:
    user_prompt = build_user_prompt(job)
    payload = {
        "model": DEEPSEEK_MODEL,
        "temperature": 0.0,
        "max_tokens": 120,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "response_format": {"type": "json_object"},
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            async with session.post(DEEPSEEK_ENDPOINT, headers=headers, json=payload) as resp:
                txt = await resp.text()
                if resp.status >= 400:
                    raise RuntimeError(f"HTTP {resp.status}: {txt[:500]}")
                data = json.loads(txt)
                content = data["choices"][0]["message"]["content"]
                obj = parse_json_object(content)
                return {
                    "canonical_entity": job["canonical_entity"],
                    "final_type": job["final_type"],
                    "canonical_key": job["canonical_key"],
                    "n_docs": job["n_docs"],
                    "n_rows": job["n_rows"],
                    "n_domains": job["n_domains"],
                    "confidence_mean": job["confidence_mean"],
                    "keep_for_analysis_input": bool(job["final_type"] in ANALYSIS_KEEP_TYPES),
                    "keep_for_analysis_llm": bool(obj.get("keep_for_analysis_llm", True)),
                    "cleaned_type_llm": str(obj.get("cleaned_type_llm", job["final_type"])),
                    "cleaned_name_llm": str(obj.get("cleaned_name_llm", job["canonical_entity"])).strip(),
                    "drop_reason_llm": str(obj.get("drop_reason_llm", "unclear")).strip(),
                    "confidence_flag_llm": str(obj.get("confidence_flag_llm", "low")).strip(),
                    "notes_llm": str(obj.get("notes_llm", "")).strip(),
                    "raw_llm_obj": obj,
                    "status": "ok",
                }
        except Exception as e:
            last_err = e
            await asyncio.sleep(min(2 ** attempt, 20))

    return {
        "canonical_entity": job["canonical_entity"],
        "final_type": job["final_type"],
        "canonical_key": job["canonical_key"],
        "n_docs": job["n_docs"],
        "n_rows": job["n_rows"],
        "n_domains": job["n_domains"],
        "confidence_mean": job["confidence_mean"],
        "keep_for_analysis_input": bool(job["final_type"] in ANALYSIS_KEEP_TYPES),
        "keep_for_analysis_llm": False,
        "cleaned_type_llm": "drop",
        "cleaned_name_llm": job["canonical_entity"],
        "drop_reason_llm": "unclear",
        "confidence_flag_llm": "low",
        "notes_llm": str(last_err)[:180] if last_err is not None else "unknown error",
        "raw_llm_obj": {},
        "status": "error",
    }


async def run_llm_cleaning(jobs: List[Dict[str, Any]], out_jsonl: str) -> None:
    timeout = aiohttp.ClientTimeout(total=HTTP_TIMEOUT)
    connector = aiohttp.TCPConnector(limit=HTTP_LIMIT, limit_per_host=HTTP_PER_HOST, ssl=False)
    queue: asyncio.Queue = asyncio.Queue(maxsize=QUEUE_SIZE)
    flush_buffer: List[str] = []
    write_lock = asyncio.Lock()

    async def writer_task():
        nonlocal flush_buffer
        while True:
            item = await queue.get()
            if item is None:
                break
            flush_buffer.append(json.dumps(item, ensure_ascii=False))
            if len(flush_buffer) >= JSONL_FLUSH_EVERY:
                async with write_lock:
                    with open(out_jsonl, "a", encoding="utf-8") as f:
                        f.write("\n".join(flush_buffer) + "\n")
                    flush_buffer = []
            queue.task_done()
        if flush_buffer:
            async with write_lock:
                with open(out_jsonl, "a", encoding="utf-8") as f:
                    f.write("\n".join(flush_buffer) + "\n")
                flush_buffer = []

    async def worker(session: aiohttp.ClientSession, subjobs: List[Dict[str, Any]], pbar):
        for job in subjobs:
            res = await call_deepseek(session, job)
            await queue.put(res)
            pbar.update(1)

    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        writer = asyncio.create_task(writer_task())
        chunks = [[] for _ in range(min(WORKERS, max(1, len(jobs))))]
        for i, job in enumerate(jobs):
            chunks[i % len(chunks)].append(job)
        with tqdm(total=len(jobs), desc="LLM canonical cleaning") as pbar:
            tasks = [asyncio.create_task(worker(session, chunk, pbar)) for chunk in chunks if chunk]
            if tasks:
                await asyncio.gather(*tasks)
        await queue.put(None)
        await writer

if llm_jobs:
    try:
        _loop = asyncio.get_running_loop()
        _has_running_loop = _loop.is_running()
    except RuntimeError:
        _has_running_loop = False
    if _has_running_loop:
        await run_llm_cleaning(llm_jobs, ENTITY_LLM_RESULTS_JSONL)
    else:
        asyncio.run(run_llm_cleaning(llm_jobs, ENTITY_LLM_RESULTS_JSONL))

print("llm cleaning jsonl:", ENTITY_LLM_RESULTS_JSONL)

existing llm-clean results: 15000
pending llm-clean jobs: 892


LLM canonical cleaning:   0%|          | 0/892 [00:00<?, ?it/s]

llm cleaning jsonl: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_llm_clean_results.jsonl


## Load and Normalize LLM Cleaning Results

In [12]:
llm_rows = []
if os.path.exists(ENTITY_LLM_RESULTS_JSONL):
    with open(ENTITY_LLM_RESULTS_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                llm_rows.append(json.loads(line))
            except Exception:
                pass

llm_results = pd.DataFrame(llm_rows)
assert len(llm_results) > 0, "No LLM cleaning results found."

llm_results["cleaned_name_llm"] = llm_results["cleaned_name_llm"].fillna("").astype(str).map(canonicalize_surface)
llm_results["cleaned_type_llm"] = llm_results["cleaned_type_llm"].fillna("drop").astype(str).str.strip().str.lower()
llm_results["keep_for_analysis_llm"] = llm_results["keep_for_analysis_llm"].fillna(False).astype(bool)
llm_results["drop_reason_llm"] = llm_results["drop_reason_llm"].fillna("unclear").astype(str)
llm_results["confidence_flag_llm"] = llm_results["confidence_flag_llm"].fillna("low").astype(str)
llm_results["llm_status"] = llm_results["status"].fillna("missing").astype(str)
llm_results = llm_results.drop_duplicates(["canonical_entity", "final_type"], keep="last").reset_index(drop=True)
llm_results.to_parquet(ENTITY_LLM_RESULTS_PARQUET, index=False)

print("llm_results shape:", llm_results.shape)
print("wrote:", ENTITY_LLM_RESULTS_PARQUET)
display(llm_results.head(20))

llm_results shape: (15892, 17)
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_llm_clean_results.parquet


,canonical_entity,final_type,canonical_key,n_docs,n_rows,n_domains,confidence_mean,keep_for_analysis_input,keep_for_analysis_llm,cleaned_type_llm,cleaned_name_llm,drop_reason_llm,confidence_flag_llm,notes_llm,raw_llm_obj,status,llm_status
0,Bing,product,bing,2223,5419,781,0.683030,False,True,product,Bing,keep,high,Microsoft's AI-powered search product,"{'canonical_entity': 'Bing', 'final_type_input...",ok,ok
1,Customers,person,customers,2051,3514,623,0.714678,True,False,drop,Customers,pronoun_generic_actor,high,"Generic actor, not specific persons","{'canonical_entity': 'Customers', 'final_type_...",ok,ok
2,AMD,company,amd,2322,8839,508,0.836175,True,True,company,AMD,keep,high,Major AI/semiconductor company,"{'canonical_entity': 'AMD', 'final_type_input'...",ok,ok
3,He,person,he,3875,7944,815,0.882384,True,False,drop,He,pronoun_generic_actor,high,"Generic pronoun, not specific person","{'canonical_entity': 'He', 'final_type_input':...",ok,ok
4,Chatbots,technology,chatbots,2724,4892,963,0.752348,True,True,technology,Chatbots,keep,high,Specific AI technology concept,"{'canonical_entity': 'Chatbots', 'final_type_i...",ok,ok
5,Experts,person,experts,2831,3795,952,0.718605,True,False,drop,Experts,pronoun_generic_actor,high,"Generic human group, not specific individuals","{'canonical_entity': 'Experts', 'final_type_in...",ok,ok
6,Singapore,government_institution,singapore,1706,3546,554,0.876075,True,True,government_institution,Singapore,keep,high,Country relevant to AI policy and investment,"{'canonical_entity': 'Singapore', 'final_type_...",ok,ok
7,Openai,company,openai,35523,141820,2548,0.792314,True,True,company,OpenAI,keep,high,"AI company, specific and analyzable","{'canonical_entity': 'Openai', 'final_type_inp...",ok,ok
8,AI Agents,technology,ai agents,2657,8561,461,0.836565,True,True,technology,AI Agents,keep,high,Specific AI technology concept,"{'canonical_entity': 'AI Agents', 'final_type_...",ok,ok
9,Developers,person,developers,3346,7730,748,0.789440,True,False,drop,Developers,pronoun_generic_actor,high,"Generic human group, not specific individuals","{'canonical_entity': 'Developers', 'final_type...",ok,ok


## Merge LLM Cleaning into Main Mention Pipeline

In [13]:
mentions_final = mentions.merge(
    llm_results[[
        "canonical_entity", "final_type", "keep_for_analysis_llm", "cleaned_type_llm",
        "cleaned_name_llm", "drop_reason_llm", "confidence_flag_llm", "llm_status"
    ]],
    on=["canonical_entity", "final_type"],
    how="left"
)

mentions_final["cleaned_name_llm"] = mentions_final["cleaned_name_llm"].fillna(mentions_final["canonical_entity"])
mentions_final["cleaned_type_llm"] = mentions_final["cleaned_type_llm"].fillna(mentions_final["final_type"])
mentions_final["keep_for_analysis_llm"] = mentions_final["keep_for_analysis_llm"].fillna(mentions_final["final_type"].isin(ANALYSIS_KEEP_TYPES))
mentions_final["drop_reason_llm"] = mentions_final["drop_reason_llm"].fillna("keep")
mentions_final["confidence_flag_llm"] = mentions_final["confidence_flag_llm"].fillna("medium")
mentions_final["llm_status"] = mentions_final["llm_status"].fillna("not_requested")

mentions_final["canonical_entity"] = mentions_final["cleaned_name_llm"].map(canonicalize_surface)
mentions_final["final_type"] = mentions_final["cleaned_type_llm"].map(lambda x: normalize_type("government institution" if x == "government_institution" else x))
mentions_final["canonical_key"] = mentions_final["canonical_entity"].map(normalize_entity_key)

mentions_final = mentions_final[
    mentions_final["final_type"].isin({"company", "technology", "government_institution", "person", "organization", "product", "other"})
].copy()
mentions_final = mentions_final[mentions_final["final_type"] != "other"].copy()
mentions_final = mentions_final[mentions_final["canonical_entity"].str.len().between(MIN_CANONICAL_LEN, MAX_CANONICAL_LEN)].copy()
mentions_final = mentions_final[~mentions_final["is_news_source"]].copy()
mentions_final = mentions_final[~mentions_final["is_numeric_junk"]].copy()
mentions_final = mentions_final[~mentions_final["is_generic_actor"]].copy()

mentions_final = mentions_final[
    ~(
        (mentions_final["final_type"] == "government_institution")
        & (mentions_final["is_low_value_gov"])
        & (~mentions_final["canonical_key"].isin({"united states", "china", "india", "united kingdom", "european union", "california", "new york", "san francisco"}))
    )
].copy()

mentions_final = mentions_final.drop_duplicates(["block_uid", "canonical_key", "final_type", "char_start", "char_end"]).reset_index(drop=True)

final_cols = [
    "doc_id", "block_id", "block_uid", "url", "date", "language", "title", "domain",
    "clean_block_text", "p_ai_block", "clean_block_len",
    "entity_surface", "entity_surface_norm", "entity_key",
    "raw_label", "raw_score", "char_start", "char_end",
    "final_type_raw", "confidence", "block_text_hash",
    "is_news_source", "is_numeric_junk",
    "canonical_entity", "final_type", "canonical_key"
]
mentions_final = mentions_final[final_cols].copy()

print("entity_mentions_final shape:", mentions_final.shape)
display(mentions_final.head(15))

entity_mentions_final shape: (3333240, 26)


,doc_id,block_id,block_uid,url,date,language,title,domain,clean_block_text,p_ai_block,...,char_start,char_end,final_type_raw,confidence,block_text_hash,is_news_source,is_numeric_junk,canonical_entity,final_type,canonical_key
0,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,...,1035,1048,person,0.903304,a75c4f305d9d5292bf75dda182b3ce1e1043f7cf,False,False,Byron Gilliam,person,byron gilliam
1,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,...,1447,1468,person,0.646446,a75c4f305d9d5292bf75dda182b3ce1e1043f7cf,False,False,Carlos Gonzalez Campo,person,carlos gonzalez campo
2,1,7,1:7,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,en,This AI video of gymnastics might be the freak...,boingboing.net,I'm sure by now you're tired of looking at ter...,0.998320,...,362,376,organization,0.856225,dbc53e7976740be3434d1d964f81473fdda217c3,False,False,Werners AI Art,organization,werners ai art
3,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,436,442,organization,0.526036,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,1min AI,organization,1min ai
4,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,481,486,product,0.770057,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,GPT-4,technology,gpt 4
5,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,488,496,organization,0.600512,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,Google AI,organization,google ai
6,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,498,508,product,0.906617,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,Gemini Pro,technology,gemini pro
7,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,581,602,product,0.596726,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,Lifetime Subscription,product,lifetime subscription
8,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,606,612,organization,0.597250,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,1min AI,organization,1min ai
9,2,7,2:7,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,"Instead, 1minAI bundles everything you need in...",0.998178,...,9,15,company,0.519907,1740a28fad1a24ef3d981c65eb838bafe2f420c5,False,False,1min AI,company,1min ai


## Build Analysis Mentions

In [14]:
analysis_mentions = mentions_final[mentions_final["final_type"].isin(ANALYSIS_KEEP_TYPES)].copy()
analysis_mentions.to_parquet(ENTITY_ANALYSIS_MENTIONS_PARQUET, index=False)
mentions_final.to_parquet(ENTITY_MENTIONS_FINAL_PARQUET, index=False)

print("wrote:", ENTITY_MENTIONS_FINAL_PARQUET)
print("wrote:", ENTITY_ANALYSIS_MENTIONS_PARQUET)
print("entity_analysis_mentions shape:", analysis_mentions.shape)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_mentions_final.parquet
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_analysis_mentions.parquet
entity_analysis_mentions shape: (2631588, 26)


## Build Analysis Summary

In [15]:
entity_analysis_summary = (
    analysis_mentions.groupby(["canonical_entity", "final_type"], dropna=False)
    .agg(
        n_docs=("doc_id", "nunique"),
        n_rows=("canonical_entity", "size"),
        n_domains=("domain", "nunique"),
        confidence_mean=("confidence", "mean"),
    )
    .reset_index()
    .sort_values(["n_docs", "n_rows", "n_domains", "confidence_mean"], ascending=[False, False, False, False])
    .reset_index(drop=True)
)
entity_analysis_summary["keep_for_analysis"] = True
entity_analysis_summary.to_parquet(ENTITY_ANALYSIS_SUMMARY_PARQUET, index=False)

print("wrote:", ENTITY_ANALYSIS_SUMMARY_PARQUET)
print("entity_analysis_summary shape:", entity_analysis_summary.shape)
display(entity_analysis_summary.head(30))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_analysis_summary.parquet
entity_analysis_summary shape: (288026, 7)


,canonical_entity,final_type,n_docs,n_rows,n_domains,confidence_mean,keep_for_analysis
0,OpenAI,company,36066,143535,2581,0.791745,True
1,Google,company,26450,95393,2436,0.830863,True
2,United States,government_institution,22616,43770,2379,0.758239,True
3,Microsoft,company,22556,74596,2131,0.833017,True
4,ChatGPT,technology,18994,57857,2567,0.744122,True
5,Generative Ai,technology,14046,44467,1690,0.824542,True
6,Machine Learning,technology,12633,28361,1841,0.823382,True
7,Nvidia,company,11569,48132,1354,0.851885,True
8,Meta,company,9346,28938,1561,0.782230,True
9,Sam Altman,person,8969,14639,1437,0.951198,True


## Build Context Table

In [16]:
context_rows = []
for (canonical_entity, final_type), sub in tqdm(
    analysis_mentions.groupby(["canonical_entity", "final_type"], dropna=False),
    desc="Building entity contexts"
):
    sub = sub.sort_values(["confidence", "clean_block_len", "doc_id", "block_id"], ascending=[False, False, True, True])
    sub = sub.drop_duplicates(["doc_id", "block_id", "block_text_hash"])
    topk = sub.head(MAX_CONTEXTS_PER_ENTITY)
    for row in topk.itertuples(index=False):
        context_rows.append({
            "canonical_entity": canonical_entity,
            "final_type": final_type,
            "domain": row.domain,
            "title": row.title,
            "date": row.date,
            "clean_block_text": compact_text_snippet(row.clean_block_text, MAX_CONTEXT_CHARS),
            "confidence": float(row.confidence),
            "doc_id": int(row.doc_id),
            "block_id": int(row.block_id),
            "url": row.url,
            "clean_block_len": int(row.clean_block_len),
        })

entity_contexts_final = pd.DataFrame(context_rows)
entity_contexts_final.to_parquet(ENTITY_CONTEXTS_FINAL_PARQUET, index=False)

print("wrote:", ENTITY_CONTEXTS_FINAL_PARQUET)
print("entity_contexts_final shape:", entity_contexts_final.shape)
display(entity_contexts_final.head(15))

Building entity contexts:   0%|          | 0/288026 [00:00<?, ?it/s]

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_contexts_final.parquet
entity_contexts_final shape: (614840, 11)


,canonical_entity,final_type,domain,title,date,clean_block_text,confidence,doc_id,block_id,url,clean_block_len
0,#17 Mike’s Famous Philly,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.477525,47114,5,https://www.businesswire.com/news/home/2024012...,351
1,#9 Club Supreme,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.745871,47114,5,https://www.businesswire.com/news/home/2024012...,351
2,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,"Super Micro Computer, Inc.: Supermicro Unveils...",2025-06-11,Delivers high-performing AI infrastructure thr...,0.454479,153982,6,https://www.finanznachrichten.de/nachrichten-2...,171
3,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,KX Launches Its First Agentic AI Blueprint for...,2025-06-11,Delivers high-performing AI infrastructure thr...,0.454228,34586,22,https://www.finanznachrichten.de/nachrichten-2...,171
4,$101m Startup,company,cityam.com,"Musicians know AI is stealing their art, but f...",2023-11-29,Stability AI exec leaves amid concerns over ‘f...,0.634083,12444,38,https://www.cityam.com/musicians-ai-artificial...,86
5,$101m Startup,company,cityam.com,"With Altman moving to Microsoft, what's next f...",2023-11-21,Stability AI exec leaves amid concerns over ‘f...,0.633668,130890,28,https://www.cityam.com/whats-next-for-openai/,86
6,$150B Valued Company,company,digitaltrends.com,OpenAI’s Sora was leaked in protest over alleg...,2024-11-26,“Hundreds of artists provide unpaid labor thro...,0.710685,141839,7,https://www.digitaltrends.com/computing/openai...,319
7,$2.855–$2.908,government_institution,mexc.co,ChatGPT’s XRP Analysis: Price Consolidates at ...,2025-09-04,ChatGPT's XRP analysis has described price con...,0.540378,108614,1,https://www.mexc.co/en-IN/news/chatgpts-xrp-an...,286
8,$2.855–$2.908,government_institution,mexc.co,ChatGPT’s XRP Analysis: Price Consolidates at ...,2025-09-04,ChatGPT's XRP analysis has described price con...,0.540349,195028,1,https://www.mexc.co/fr/news/chatgpts-xrp-analy...,286
9,$3.30–3.70,government_institution,mexc.fm,ChatGPT-5 Shares Smart XRP Trading Strategy Fo...,2025-09-19,"For the past few months, XRP price has been st...",0.467257,15480,0,https://www.mexc.fm/en-TR/news/chatgpt-5-share...,3793


## Quick Readout

In [17]:
final_mentions_preview = pd.read_parquet(ENTITY_MENTIONS_FINAL_PARQUET)
analysis_mentions_preview = pd.read_parquet(ENTITY_ANALYSIS_MENTIONS_PARQUET)
analysis_summary_preview = pd.read_parquet(ENTITY_ANALYSIS_SUMMARY_PARQUET)
contexts_preview = pd.read_parquet(ENTITY_CONTEXTS_FINAL_PARQUET)
candidate_pool_preview = pd.read_parquet(CANDIDATE_POOL_FOR_LLM_PARQUET)

print("entity_mentions_final shape:", final_mentions_preview.shape)
print("entity_analysis_mentions shape:", analysis_mentions_preview.shape)
print("entity_analysis_summary shape:", analysis_summary_preview.shape)
print("entity_contexts_final shape:", contexts_preview.shape)
print("candidate_pool shape:", candidate_pool_preview.shape)

print("\nfinal type distribution:")
display(final_mentions_preview["final_type"].value_counts(dropna=False))

print("\nanalysis type distribution:")
display(analysis_mentions_preview["final_type"].value_counts(dropna=False))

print("\nconfidence summary in final mentions:")
display(final_mentions_preview["confidence"].describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]))

print("\nnews-source rows in final mentions:")
print(int(final_mentions_preview["is_news_source"].sum()))

print("\nnumeric-junk rows in final mentions:")
print(int(final_mentions_preview["is_numeric_junk"].sum()))

print("\ntop entities by document coverage:")
display(analysis_summary_preview.head(30))

print("\nexample mentions:")
display(final_mentions_preview[["doc_id", "block_id", "domain", "canonical_entity", "final_type", "confidence", "clean_block_text"]].head(15))

print("\nexample contexts:")
display(contexts_preview.head(15))

print("\ncandidate pool preview:")
display(candidate_pool_preview.head(30))

entity_mentions_final shape: (3333240, 26)
entity_analysis_mentions shape: (2631588, 26)
entity_analysis_summary shape: (288026, 7)
entity_contexts_final shape: (614840, 11)
candidate_pool shape: (15000, 8)

final type distribution:


,count
final_type,
company,1196610
technology,608860
person,471851
product,373791
government_institution,354267
organization,327861



analysis type distribution:


,count
final_type,
company,1196610
technology,608860
person,471851
government_institution,354267



confidence summary in final mentions:


...

,doc_id,block_id,domain,canonical_entity,final_type,confidence,clean_block_text
0,0,0,blockworks.co,Byron Gilliam,person,0.903304,The price of BAD is down -0.61% since last hou...
1,0,0,blockworks.co,Carlos Gonzalez Campo,person,0.646446,The price of BAD is down -0.61% since last hou...
2,1,7,boingboing.net,Werners AI Art,organization,0.856225,I'm sure by now you're tired of looking at ter...
3,2,6,boingboing.net,1min AI,organization,0.526036,AI isn't going anywhere. The problem is that w...
4,2,6,boingboing.net,GPT-4,technology,0.770057,AI isn't going anywhere. The problem is that w...
5,2,6,boingboing.net,Google AI,organization,0.600512,AI isn't going anywhere. The problem is that w...
6,2,6,boingboing.net,Gemini Pro,technology,0.906617,AI isn't going anywhere. The problem is that w...
7,2,6,boingboing.net,Lifetime Subscription,product,0.596726,AI isn't going anywhere. The problem is that w...
8,2,6,boingboing.net,1min AI,organization,0.597250,AI isn't going anywhere. The problem is that w...
9,2,7,boingboing.net,1min AI,company,0.519907,"Instead, 1minAI bundles everything you need in..."



example contexts:


,canonical_entity,final_type,domain,title,date,clean_block_text,confidence,doc_id,block_id,url,clean_block_len
0,#17 Mike’s Famous Philly,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.477525,47114,5,https://www.businesswire.com/news/home/2024012...,351
1,#9 Club Supreme,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.745871,47114,5,https://www.businesswire.com/news/home/2024012...,351
2,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,"Super Micro Computer, Inc.: Supermicro Unveils...",2025-06-11,Delivers high-performing AI infrastructure thr...,0.454479,153982,6,https://www.finanznachrichten.de/nachrichten-2...,171
3,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,KX Launches Its First Agentic AI Blueprint for...,2025-06-11,Delivers high-performing AI infrastructure thr...,0.454228,34586,22,https://www.finanznachrichten.de/nachrichten-2...,171
4,$101m Startup,company,cityam.com,"Musicians know AI is stealing their art, but f...",2023-11-29,Stability AI exec leaves amid concerns over ‘f...,0.634083,12444,38,https://www.cityam.com/musicians-ai-artificial...,86
5,$101m Startup,company,cityam.com,"With Altman moving to Microsoft, what's next f...",2023-11-21,Stability AI exec leaves amid concerns over ‘f...,0.633668,130890,28,https://www.cityam.com/whats-next-for-openai/,86
6,$150B Valued Company,company,digitaltrends.com,OpenAI’s Sora was leaked in protest over alleg...,2024-11-26,“Hundreds of artists provide unpaid labor thro...,0.710685,141839,7,https://www.digitaltrends.com/computing/openai...,319
7,$2.855–$2.908,government_institution,mexc.co,ChatGPT’s XRP Analysis: Price Consolidates at ...,2025-09-04,ChatGPT's XRP analysis has described price con...,0.540378,108614,1,https://www.mexc.co/en-IN/news/chatgpts-xrp-an...,286
8,$2.855–$2.908,government_institution,mexc.co,ChatGPT’s XRP Analysis: Price Consolidates at ...,2025-09-04,ChatGPT's XRP analysis has described price con...,0.540349,195028,1,https://www.mexc.co/fr/news/chatgpts-xrp-analy...,286
9,$3.30–3.70,government_institution,mexc.fm,ChatGPT-5 Shares Smart XRP Trading Strategy Fo...,2025-09-19,"For the past few months, XRP price has been st...",0.467257,15480,0,https://www.mexc.fm/en-TR/news/chatgpt-5-share...,3793



candidate pool preview:


,canonical_entity,final_type,n_docs,n_rows,n_domains,confidence_mean,keep_for_analysis,canonical_key
0,AI,technology,112374,730428,4673,0.880893,True,ai
1,Artificial Intelligence,technology,52820,99056,3803,0.821245,True,artificial intelligence
2,OpenAI,company,28616,95960,2316,0.806287,True,open ai
3,Google,company,26404,95115,2435,0.831170,True,google
4,Microsoft,company,22494,74173,2130,0.834231,True,microsoft
5,ChatGPT,product,21637,59950,2359,0.795421,False,chat gpt
6,United States,government_institution,16964,31739,2054,0.764726,True,united states
7,ChatGPT,technology,15114,44034,2363,0.759445,True,chat gpt
8,OpenAI,organization,14881,44597,1894,0.762220,False,open ai
9,Generative Ai,technology,12127,35582,1609,0.830927,True,generative ai
